# HCCL

本节回答本章的第二个问题：**HCCL 到底是什么、由哪些概念组成？**

先看它的定义与两大核心特点，再看它「分层解耦」的开放生态，
然后理清四个基础概念，最后补两块背景，一是**超节点与集群的互联带宽层级**，二是**框架是怎么用上 HCCL 的**。

## 本节学习大纲

- HCCL 是什么（定义 + 两大核心特点）
- HCCL 开放生态：分层解耦，全栈开源
- HCCL 基础概念：通信成员 / 通信域 / 通信算子 / 通信算法
- 背景补充：超节点是什么，以及集群的互联带宽层级（为什么 HCCL 要「感知拓扑」）
- 框架是怎么用上 HCCL 的
- 课堂小练
- 本节总结
- 课后延伸

## 1. HCCL 是什么

> **HCCL：CANN 的集合通信库，为昇腾计算集群提供高性能集合通信服务。**

**HCCL（Huawei Collective Communication Library）** 是华为集合通信库，是 CANN 的基础组件之一。
它依托昇腾芯片高效的通信引擎与总线 / 网络协议，为昇腾计算集群提供**高性能、高可靠、高易用**的集合通信解决方案。

### 1.1 两大核心特点

**① 高性能通信算法**

- 拓扑感知的高性能通信算法，为**不同业务场景、不同拓扑架构**下的计算集群提供高效通信方案。

> 为什么强调「拓扑感知」？因为集群里数据能走的「路」不止一种，而**不同链路的带宽和时延
> 差得很远**。算法必须知道数据实际走的是哪条路，才能决定「先在本机内归约、再往外传」，
> 还是「直接往外传」，从而避免把宝贵的跨机带宽浪费在本来可以在本地完成的计算上。
> （超节点是什么、各条链路具体差多少，见本节 §4「超节点与集群的互联带宽层级」。）

**② 硬化调度 + 独立通信加速引擎**

- 硬化调度和通信原语，**降低调度开销**，精准控制系统抖动，提升计算集群算力利用率；
- 独立通信引擎，支持**随路计算**，不占用计算核，支持**计算与通信高效并发**。

> 「硬化调度」指的是把通信任务的调度逻辑从 CPU 软件下沉到硬件完成，省掉了 CPU 反复下发指令的
> 开销，也消除了软件调度带来的抖动。「独立通信引擎」则意味着通信由专门的硬件单元负责搬运，
> 计算核（AI Core）可以继续算下一个算子——这正是后面「计算通信并发流水」的基础。

## 2. HCCL 开放生态：分层解耦，全栈开源

HCCL 并非一个庞大单体，而是按**分层解耦**的思路拆成多个开源仓库，各自职责清晰。

![图 1](./images/png/cann_software_stack.png)

*图 1 · CANN 软件栈分层。HCCL 位于「通信库」层，与 HCOMM 分层解耦。*

### 2.1 分层结构

| 层次 | 组件 | 职责 |
| --- | --- | --- |
| **训推框架接入层** | PyTorch 等 | 上层 AI 框架接入，调用通信算子 |
| **HCCL 仓** | HCCL 集合通信库 | 集合通信库 · **算法 / 算子** |
| **HCOMM 仓** | 通信基础库 | **通信域 / 基础通信能力** |
| **RTS / DRV** | 驱动层 | 芯片通信能力 |

> **分层解耦的核心收益**：HCCL 仓专注「业务场景」（算子与算法应怎么设计），HCOMM 仓专注
> 「基础通信能力」（通信域如何管理、链路如何打通）。两者通过稳定接口衔接，任何一层升级都不
> 会强绑定另一层，也让社区开发者可以按需参与其中一层。

### 2.2 三大社区入口

| 入口 | 地址 | 内容 |
| --- | --- | --- |
| **HCCL 仓** | `gitcode.com/cann/hccl` | 集合通信库 · 算法 / 算子（已开源） |
| **HCOMM 仓** | `gitcode.com/cann/hcomm` | 通信基础库 · 基础通信能力（已开源） |
| **HCCL SIG 组** | `gitcode.com/cann/community` | 社区需求 / 讨论 / 活动运营 |

> **HCCL 仓 和 HCOMM 仓，实现业务场景与基础通信能力分层解耦。**

## 3. HCCL 基础概念

在深入并行方式和具体通信操作之前，先把 HCCL 里四个最基础的概念理清：
**通信成员、通信域、通信算子、通信算法**。它们是从「谁在通信」到「怎么通信」的层层递进关系。

![图 2](./images/png/comm_domain_model.png)

*图 2 · 通信成员 → 通信域 → 通信算子 → 通信算法 的层次关系。*

### 3.1 四个核心概念

| 概念 | 定义 | 关键点 |
| --- | --- | --- |
| **通信成员** | 参与通信的**最小逻辑实体**，可以是一颗芯片、一个 Die、甚至一个进程 | 一般称为 **Rank** |
| **通信域** | 一组通信成员的组合，描述**通信范围** | 一个集群可建多个域；一个成员可加入多个域 |
| **通信算子** | 在通信域内**完成通信任务**的算子 | 集合通信 = 所有成员一起参与；域内算子**串行执行** |
| **通信算法** | 完成某个算子的**实现方案** | 不同拓扑 / 数据量用不同算法，实现不同**流量规划策略** |

### 3.2 用一个例子串起来

假设一个 4 卡集群要执行一次 `AllReduce`：

1. 4 张卡各自是**通信成员**（Rank0 ~ Rank3）；
2. 它们组成一个**通信域**，约定了「谁来参加这次通信」；
3. 它们调用 `AllReduce` 这个**通信算子**，算子规定了「所有 Rank 都拿到归约结果」这一语义；
4. 底层可以选择 **Ring**、**Mesh** 等不同**通信算法**去实现它——走哪条链路、分几步交换，
   由算法决定。

> **算子规定「结果」，算法规定「过程」。** 这是理解 HCCL 分层设计的关键：
> 上层调用者只关心算子的语义，不必关心底层用哪种算法；而库的实现者则针对不同拓扑，
> 为同一个算子提供多种算法，最大化集群通信性能。

> **八个常见集合通信算子**（Gather / Reduce / Broadcast / Scatter / AllReduce / AllGather /
> ReduceScatter / AlltoAll）的逐个详解，放在**第 2 章《HCCL 典型算子与算法》的 2.2 节**——
> 那里有完整的「操作前 → 操作后」对照图和易混点辨析。本节先把「通信算子」这个概念讲清楚就够了。

## 4. 背景补充：超节点与集群的互联带宽层级

> 这一节是**背景补充**，用来把 §1.1 里「**拓扑感知**」这个词讲明白。
> 本节引用的数字全部来自**华为官方**（2026-07 Atlas 950 超节点亮相、2026-09 全联接大会「灵衢互联」发布）。

### 4.1 先搞清楚：什么是「超节点」

**一句话**：超节点就是用一条高速总线，把**多个机柜**的 NPU 直接连成**一个逻辑设备**——对外看起来「像一台机器」。

| | 传统集群 | 超节点 |
| --- | --- | --- |
| 卡之间怎么通信 | 过交换机、**走网络**（转发式） | **免交换直连**（总线式） |
| 内存 | 各自独立，访问别的卡要显式拷贝 | **超节点内全局统一编址**，可以直接 load / store |

> 华为官方对超节点的描述：通过自研的灵衢互联协议，将多个分布式设备深度互联，
> 「**在逻辑上形成像『一台机器』一样的整体**」，具备**大带宽、低时延、统一内存编址**三大核心特点。
> 官方还举了个最直白的例子：Atlas 650E 两台机器灵衢直连，16 个 NPU 免交换互联，
> 「**两台机器像一台运行**」——这就构成一个**小型超节点**。

**⚠️ 两个最容易搞错的地方，请务必记住：**

1. **超节点本身就是跨机柜的。** Atlas 950 基准配置 **1024 卡 = 16 个机柜**，最大 **8192 卡 = 128 个机柜**
   （官方另提到 4096 卡超节点）。所以「**跨柜**」不等于「**出了超节点**」——柜与柜之间可能还在同一个超节点里面。
2. **它不是单纯为了堆规模，而是为了拆掉「通信墙」。** 官方原话：
   **「十万卡集群的实际模型算力利用率仅约 20%，大量算力处于等待通信完成的状态。」**
   卡越多、通信越容易拖后腿，这正是 §1.5 讲的那个新瓶颈。

### 4.2 集群里数据能走哪几种「路」：灵衢三级互联

官方把互联设备做成「**分层分级**」，覆盖**柜内、跨柜与集群三级互联**：

| 链路 | 连接谁 | 带宽 / 时延 |
| --- | --- | --- |
| **① 超节点内 · 柜内**（灵衢互联刀片） | 同一机柜内，板与板之间 | **零电缆**、零电路损耗，时延最低 |
| **② 超节点内 · 跨柜**（灵衢互联设备） | **同一个超节点内**，柜与柜 | **176 个端口**、每端口 **1.6T**、单机 **280T** 全光互联，RTT 低至 **2 微秒** |
| **③ 超节点之间 · 集群**（UBG 灵衢交换机） | 超节点 ↔ 超节点 | Radix 高达 **1024**，可支撑**百万卡**集群 |
| **④ PCIe** | Host ↔ Device | 主机侧总线，速率**远低于**上面三者 |

> 官方原话：柜内灵衢互联刀片「实现零电缆、零电路损耗，一个 4096 超节点可节省 196 公里铜缆」；
> 跨柜灵衢互联设备「支持 176 个端口，每端口达 1.6T，单机 280T 全光互联……RTT 时延低至 2 微秒」；
> UBG 灵衢网络交换机「具备高达 1024 的超大 Radix 扇出能力，可支撑百万卡超节点集群」。
> 整体口径：互联带宽「从 **百 GB 级**提升至 **TB 级**」，RTT 时延「从 **7 微秒**压缩至 **2 微秒**」。

一句话记住这张表：**① 最快，② 次之，③ 再降一档（但仍远快于传统网络），④ 是主机侧总线。**

### 4.3 这件事为什么决定算法怎么选

同一条数据，走「超节点内」还是走「超节点之间」，**带宽差一个数量级、时延差好几倍**。
这直接带来两个后果：

1. **能在本地做的，绝不往外发。** 例如一次 AllReduce：如果先把同一超节点内几张卡的数据
   在本地归约掉、再把结果往外传，跨机链路上搬运的数据量就小得多——**宝贵的跨机带宽省下来了**。
2. **算法必须按拓扑来选。** 第 2 章会看到：Ring 用 N-1 步逐跳传，对拓扑要求低；
   Mesh 只在 Full-mesh 下成立、1 步完成；Halving-Doubling 用 log₂N 步换取更大的访问步长。
   选哪个，本质就是在**给定拓扑下做带宽与时延的权衡**。

**HCCL 正是按上面这三级来分级选算法的**——三个层级各选各的算法，再拼起来：

| HCCL 层级（`HCCL_ALGO` 里的名字） | 对应哪条「路」 | 这一层用什么算法 |
| --- | --- | --- |
| **Server 内**（level0） | ① 柜内 | Mesh / Ring / Double-Ring / Star，**由硬件拓扑自动选择，用户不可配置** |
| **Server 间**（level1） | ② 跨柜 | Ring / RHD / NHR / NB / AHC / Pipeline / Pairwise，**自适应选择** |
| **超节点间**（level2） | ③ 集群 | 同一套算法自适应；**默认 RHD**（超节点个数 < 8 且非 2 的整数次幂时用 ring） |

> 说明：HCCL 文档里的三级叫「**节点内 / 节点间 / 超节点间**」，其中「节点」指一台服务器（约 8 张卡）。
> 它与灵衢的「柜内 / 跨柜 / 集群」**大致对应**：节点内就是柜内那个最高速的域，节点间则可能落在
> 「同柜跨机」或「跨柜」这两种情况上。两者颗粒度略有差别，不必死抠。

三个补充事实：

- **level0 用户管不了**：`HCCL_ALGO` 里的 level0 只能填 `NA`，柜内走什么算法由硬件拓扑决定；
- **level1 / level2 可以人工指定**：例如 `HCCL_ALGO="level0:NA;level1:NHR;level2:H-D_R"`，
  还能按算子分别配置；**一旦指定，自适应选择就不再生效**；
- **Atlas 950 系列**（Ascend 950PR / 950DT）的 level1 **默认就是 NHR**。

更关键的是：**HCCL 不是「三级里挑一级走」，而是三级分工、分级执行**。以 AllReduce 为例：

```text
超节点内 ReduceScatter  →  超节点间 AllReduce  →  超节点内 AllGather
```

也就是说，**大数据量的搬运放在带宽更高的层**，跨集群链路只搬小结果。
这正是上面第 1 条「能在本地做的绝不往外发」在算法层面的落地。

一句话：**HCCL 要「感知拓扑」，是因为不同的路快慢差太多；把数据送到它该走的路上，才叫高性能。**

## 5. 框架是怎么用上 HCCL 的

回到 §2 那张软件栈图：HCCL 的上面一层是「**训推框架接入**」——PyTorch 等框架负责调用通信算子。
这一节回答一个问题：**框架是怎么把用户写的「并行策略」变成 HCCL 调用的？**

### 5.1 用户写的是「策略」，不是「通信」

以最常用的数据并行为例，用户只需要写一行：

```python
model = DistributedDataParallel(model)
```

**不用手写任何通信代码**。反向传播一结束，框架会**自动**发起一次 **AllReduce**，
把各卡的梯度同步成一致的结果，然后各卡才更新权重。

### 5.2 换一种并行方式，框架插入的算子就变了

数据并行同步的是梯度（AllReduce）；换成张量并行，框架会在层内插入 AllReduce / AllGather
把结果拼回；换成专家并行，插入的是 All2All，用来把 token 分发到专家所在的卡、再把结果收回来。
**每个算子具体怎么用，第 2 章会逐个展开。**

### 5.3 这就是 HCCL 的价值

用户面对的是「并行策略」，HCCL 面对的是「通信算子」，中间由**框架**负责翻译。

**训推框架无需关心底层链路，直接调用即可获得高效通信。** ——这正是 §2 里「分层解耦」的意义：
框架只管调用，HCCL 只管把这次调用在底层链路上跑得足够快。

## 6. 课堂小练

请根据本节学习内容完成以下题目进行自测。

**1、HCCL 是什么？**

---

**2、（多选题）HCCL（华为集合通信库）的主要特点有哪些？**

- **A)** 内置高性能集合通信算法，提升大规模并行计算通信效率
- **B)** 计算通信统一硬化调度，降低调度开销，优化硬件资源利用率
- **C)** 计算通信高性能并发，计算与通信并发流水执行，系统性能进一步提升

---

**3、（判断题）** 昇腾集群里，数据一旦要「跨柜」，就说明它已经出了超节点、得走集群网络了。

---

**4、（简答题）** 用数据并行训练时，用户需要自己写 AllReduce 的调用代码吗？框架是怎么处理的？

**执行以下代码获取答案。**

In [ ]:
# 读取本节课堂小练答案（用 Python 读文件：跨平台，并保证中文不乱码）
import pathlib
_p = pathlib.Path('./answer/01.03_answer.txt')
if _p.exists():
    print(_p.read_text(encoding='utf-8'))
else:
    print(f'未找到答案文件：{_p}')
    print('（请确认 answer/ 目录与本 notebook 在同一层）')

## 7. 本节总结

### HCCL 是什么

- **华为集合通信库**（Huawei Collective Communication Library），CANN 的基础组件之一；
- **使用场景**：分布式集群训练、推理等；
- **两大特点**：① 高性能通信算法（拓扑感知）；② 硬化调度 + 独立通信加速引擎
  （低开销、低抖动，随路计算、计算通信并发）；
- **开放生态**：HCCL 仓（算法 / 算子）+ HCOMM 仓（通信域 / 基础通信能力）分层解耦，全栈开源。

### HCCL 基础概念

- **通信成员（Rank）**：参与通信的最小逻辑实体（芯片 / Die / 进程）；
- **通信域**：一组通信成员的组合，描述通信范围；一个集群可有多个域，一个成员可加入多个域；
- **通信算子**：在通信域内完成通信任务的算子，域内算子**串行执行**；
- **通信算法**：算子的具体实现方案，决定链路与分步；
- **一句话**：**算子规定「结果」，算法规定「过程」。**

### 超节点与互联带宽（背景补充）

- **超节点**：用灵衢总线把**多个机柜**的 NPU 连成**一个逻辑设备**，对外「像一台机器」；
  三大特征：大带宽、低时延、**超节点内全局内存统一编址**。**超节点本身就是跨机柜的**
  （Atlas 950 基准 1024 卡 = 16 个机柜，最大 8192 卡 = 128 个机柜）；
- 官方把互联分成**三级**：**柜内**（灵衢互联刀片，零电缆）→ **跨柜**（灵衢互联设备，
  176 端口 × 1.6T、单机 280T 全光）→ **集群**（UBG 灵衢交换机，Radix 1024，支撑百万卡）；
  另有 **PCIe** 用于 Host ↔ Device。官方口径：带宽从**百 GB 级 → TB 级**，RTT 从 **7 微秒 → 2 微秒**；
- **HCCL 就按这三级分级选算法**：Server 内（level0，自动、不可配）/ Server 间（level1）/
  超节点间（level2），后两级自适应，也可用 `HCCL_ALGO` 强制指定。一次 AllReduce =
  **内 ReduceScatter → 间 AllReduce → 内 AllGather**（大数据量放在带宽更高的层）；
- 所以算法必须**感知拓扑**：能在本地归约的绝不往外发；算法选型本质是**带宽与时延的权衡**。

### 框架是怎么用上 HCCL 的

- 用户写的是**并行策略**（如 `DistributedDataParallel`），**不用手写通信代码**；
- 框架在合适时机**自动插入**通信算子（DP → AllReduce；TP → AllReduce / AllGather；EP → All2All）；
- **训推框架无需关心底层链路，直接调用即可获得高效通信**——这就是分层解耦的意义。

### 八个常见集合通信算子

> 详解见**第 2 章 2.2 节**（含「操作前 → 操作后」对照图与易混点辨析）。
> 本节只需记住：按参与关系分成 **多对一 / 一对多**（有 root）与 **多对多**（无 root）两大家族。

## 8. 课后延伸

| 资源 | 地址 |
| --- | --- |
| HCCL 仓（集合通信库 · 算法 / 算子） | https://gitcode.com/cann/hccl |
| HCOMM 仓（通信基础库） | https://gitcode.com/cann/hcomm |
| HCCL SIG 组（社区需求 / 讨论 / 活动） | https://gitcode.com/cann/community |
| CANN 学习中心 | https://gitcode.com/cann/cann-learning-hub |

<img src="./images/qrcode_cann_community.jpg" alt="CANN 社区" width="18%">

### 下一章预告

**第 2 章 · HCCL 典型算子与算法**：本章提到的 AllReduce、All2All 等算子究竟是如何实现的？
Ring、Mesh、Halving-Doubling 算法各自适合什么拓扑？下一章将逐一展开。